In [50]:
import time

import logging
logging.basicConfig(level=logging.INFO)

logger = logging.getLogger("test")
logger.setLevel(logging.INFO)

import torch
from torch.amp import autocast

from contextlib import nullcontext
from pytorch_forecasting.layers._attention._full_attention import FullAttention, _FullAttention


In [42]:
BATCH_SIZE = 32
NUM_HEADS = 8
SEQ_LEN = 720
D_MODEL = 256
USE_MASK = False
DROPOUT_RATE = 0.0

In [44]:
class TestAttentionBackendsCUDA:

    def test_cuda_no_amp(self) -> None:
        _benchmark_attention_backends(
            batch_size=BATCH_SIZE,
            num_heads=NUM_HEADS,
            seq_len=SEQ_LEN,
            d_model=D_MODEL,
            use_mask=USE_MASK,
            dropout_rate=DROPOUT_RATE,
            device_name="cuda",
            use_amp=False,
        )

    def test_cuda_amp_bfloat16(self) -> None:
        _benchmark_attention_backends(
            batch_size=BATCH_SIZE,
            num_heads=NUM_HEADS,
            seq_len=SEQ_LEN,
            d_model=D_MODEL,
            use_mask=USE_MASK,
            dropout_rate=DROPOUT_RATE,
            device_name="cuda",
            use_amp=True,
            amp_dtype=torch.bfloat16,
        )

    def test_cuda_amp_float16(self) -> None:
        _benchmark_attention_backends(
            batch_size=BATCH_SIZE,
            num_heads=NUM_HEADS,
            seq_len=SEQ_LEN,
            d_model=D_MODEL,
            use_mask=USE_MASK,
            dropout_rate=DROPOUT_RATE,
            device_name="cuda",
            use_amp=True,
            amp_dtype=torch.float16,
        )


class TestAttentionBackendsCPU:

    def test_cpu_no_amp(self) -> None:
        _benchmark_attention_backends(
            batch_size=BATCH_SIZE,
            num_heads=NUM_HEADS,
            seq_len=SEQ_LEN,
            d_model=D_MODEL,
            use_mask=USE_MASK,
            dropout_rate=DROPOUT_RATE,
            device_name="cpu",
            use_amp=False,
        )

    def test_cpu_amp_bfloat16(self) -> None:
        _benchmark_attention_backends(
            batch_size=BATCH_SIZE,
            num_heads=NUM_HEADS,
            seq_len=SEQ_LEN,
            d_model=D_MODEL,
            use_mask=USE_MASK,
            dropout_rate=DROPOUT_RATE,
            device_name="cpu",
            use_amp=True,
            amp_dtype=torch.bfloat16,
        )

    def test_cpu_amp_float16(self) -> None:
        _benchmark_attention_backends(
            batch_size=BATCH_SIZE,
            num_heads=NUM_HEADS,
            seq_len=SEQ_LEN,
            d_model=D_MODEL,
            use_mask=USE_MASK,
            dropout_rate=DROPOUT_RATE,
            device_name="cpu",
            use_amp=True,
            amp_dtype=torch.float16,
        )


def _benchmark_attention_backends(
    batch_size: int,
    num_heads: int,
    seq_len: int,
    d_model: int,
    use_mask: bool = False,
    dropout_rate: float = 0.0,
    device_name: str = "cuda",
    use_amp: bool = False,
    amp_dtype: torch.dtype = torch.bfloat16,
    num_warmup: int = 3,
    num_runs: int = 10,
) -> None:
    """Benchmark einsum vs pytorch attention backends.

    Args:
        batch_size (int): Batch size
        num_heads (int): Number of attention heads
        seq_len (int): Sequence length
        d_model (int): Model dimension
        use_mask (int): Whether to use causal masking
        dropout_rate (float): Dropout rate
        device (str): Device to run on ("cuda" or "cpu")
        use_amp (bool): Whether to use automatic mixed precision
        amp_dtype (torch.dtype): AMP dtype (torch.float16 or torch.bfloat16)
        num_warmup (int): Number of warmup runs
        num_runs (int): Number of benchmark runs
    """

    device: torch.device = torch.device(device_name)
    head_dim = d_model // num_heads

    # The test tensors (given the args) must replicate the shapes we will have
    # in the actual models
    query = torch.randn(batch_size, seq_len, num_heads, head_dim).to(device)
    key = torch.randn(batch_size, seq_len, num_heads, head_dim).to(device)
    value = torch.randn(batch_size, seq_len, num_heads, head_dim).to(device)

    # The einsum backend is meant to be slower, but more flexible to allow us
    # inspecting attention scores in experimentation -- with this benchmark /
    # test what we need to achieve is to port to an optimized attn
    # implementation with tangible improvement, yet make sure that our outputs
    # are the same
    attn_einsum = _FullAttention(
        mask_flag=use_mask,
        attention_dropout=dropout_rate,
    ).to(device)

    attn_pytorch = FullAttention(
        mask_flag=use_mask,
        attention_dropout=dropout_rate,
    ).to(device)

    # AMP context and scaler
    amp_context = (
        autocast(device_type=device.type, dtype=amp_dtype)
        if use_amp
        else nullcontext()
    )

    # Warmup epochs -- these are needed for GPU specifically
    with amp_context:
        for _ in range(num_warmup):
            _, _ = attn_einsum(query, key, value, attn_mask=None)
            _, _ = attn_pytorch(query, key, value, attn_mask=None)

    # Forward pass benchmarking - einsum
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    start = time.time()
    with amp_context:
        for _ in range(num_runs):
            out_einsum, _ = attn_einsum(query, key, value, attn_mask=None)
            if device.type == "cuda":
                torch.cuda.synchronize()
    end = time.time()
    time_einsum = (end - start) / num_runs
    mem_einsum = (
        torch.cuda.max_memory_allocated() / 1024**2
        if device.type == "cuda"
        else 0.0
    )

    # Forward pass benchmarking - pytorch
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    start = time.time()
    with amp_context:
        for _ in range(num_runs):
            out_pytorch, _ = attn_pytorch(query, key, value, attn_mask=None)
            if device.type == "cuda":
                torch.cuda.synchronize()
    end = time.time()
    time_pytorch = (end - start) / num_runs
    mem_pytorch = (
        torch.cuda.max_memory_allocated() / 1024**2
        if device.type == "cuda"
        else 0.0
    )

    # Equivalence check
    # Here I just compare the last outputs from the loops, because we are
    # looping over the same input anyways

    # The datatypes, given their precision have different absolute and relative
    # tolerances for difference
    dtype = out_einsum.dtype
    if dtype == torch.float32:
        rtol, atol = 1e-5, 1e-5
    elif dtype == torch.float16:
        rtol, atol = 1e-2, 5e-3
    elif dtype == torch.bfloat16:
        rtol, atol = 2e-2, 1e-2
    else:
        rtol, atol = 1e-5, 1e-5

    is_close = torch.allclose(out_einsum, out_pytorch, rtol=rtol, atol=atol)
    max_diff = (out_einsum - out_pytorch).abs().max().item()
    mean_diff = (out_einsum - out_pytorch).abs().mean().item()

    # Backward pass benchmarking
    query.requires_grad = True
    key.requires_grad = True
    value.requires_grad = True

    # Backward - einsum
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    start = time.time()
    with amp_context:
        out_einsum, _ = attn_einsum(query, key, value, attn_mask=None)
        loss_einsum = out_einsum.sum()

    loss_einsum.backward()

    if device.type == "cuda":
        torch.cuda.synchronize()
    time_einsum_bwd = time.time() - start
    mem_einsum_bwd = (
        torch.cuda.max_memory_allocated() / 1024**2
        if device.type == "cuda"
        else 0.0
    )

    assert query.grad is not None
    assert key.grad is not None
    assert value.grad is not None

    grad_q_einsum = query.grad.clone()
    grad_k_einsum = key.grad.clone()
    grad_v_einsum = value.grad.clone()

    query.grad = None
    key.grad = None
    value.grad = None

    # Backward - pytorch
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    start = time.time()
    with amp_context:
        out_pytorch, _ = attn_pytorch(query, key, value, attn_mask=None)
        loss_pytorch = out_pytorch.sum()

    loss_pytorch.backward()

    if device.type == "cuda":
        torch.cuda.synchronize()
    time_pytorch_bwd = time.time() - start
    mem_pytorch_bwd = (
        torch.cuda.max_memory_allocated() / 1024**2
        if device.type == "cuda"
        else 0.0
    )

    assert query.grad is not None
    assert key.grad is not None
    assert value.grad is not None

    grad_q_pytorch = query.grad.clone()
    grad_k_pytorch = key.grad.clone()
    grad_v_pytorch = value.grad.clone()

    # Print results
    logger.info("=" * 80)
    logger.info(
        f"CONFIG: batch={batch_size}, heads={num_heads}, seq={seq_len}, "
        f"dim={d_model}"
    )
    logger.info(
        f"Device: {device}, AMP: {use_amp} ({amp_dtype if use_amp else 'N/A'})"
    )
    logger.info("=" * 80)

    logger.info("EQUIVALENCE CHECK")
    logger.info(f"Dtype: {dtype}")
    logger.info(f"Tolerance: rtol={rtol}, atol={atol}")
    logger.info(f"All close: {is_close}")
    logger.info(f"Max diff: {max_diff:.6e}")
    logger.info(f"Mean diff: {mean_diff:.6e}")
    logger.info("=" * 80)

    logger.info("FORWARD PASS")
    logger.info(
        f"Einsum backend: {time_einsum*1000:.2f} ms | {mem_einsum:.2f} MB"
    )
    logger.info(
        f"PyTorch backend: {time_pytorch*1000:.2f} ms | {mem_pytorch:.2f} MB"
    )
    logger.info(f"Speedup: {time_einsum/time_pytorch:.2f}x")
    if device.type == "cuda":
        logger.info(
            f"Memory saved: {mem_einsum - mem_pytorch:.2f} MB "
            f"({(1 - mem_pytorch/mem_einsum)*100:.1f}%)"
            if mem_einsum > 0
            else "Memory saved: N/A"
        )
    logger.info("=" * 80)

    logger.info("BACKWARD PASS")
    logger.info(
        f"Einsum: {time_einsum_bwd*1000:.2f} ms | {mem_einsum_bwd:.2f} MB"
    )
    logger.info(
        f"PyTorch: {time_pytorch_bwd*1000:.2f} ms | {mem_pytorch_bwd:.2f} MB"
    )
    logger.info(f"Speedup: {time_einsum_bwd/time_pytorch_bwd:.2f}x")
    if device.type == "cuda":
        logger.info(f"Memory saved: {mem_einsum_bwd - mem_pytorch_bwd:.2f} MB")
    logger.info("=" * 80)

    assert (
        is_close
    ), f"Forward pass outputs not close. Max diff: {max_diff:.6e}, Mean diff: {mean_diff:.6e}"

    # Looser tolerance for gradient equivalence:
    if dtype == torch.float32:
        # Standard single-precision tolerances remain tight
        rtol_grad, atol_grad = 1e-5, 1e-5
    elif dtype == torch.float16:
        # Loosest recommended tolerances for f16 gradients
        # If this fails, the error is likely a functional bug.
        rtol_grad, atol_grad = 1e-2, 5e-2
    elif dtype == torch.bfloat16:
        # Loosest recommended tolerances for bf16 gradients (for safety)
        rtol_grad, atol_grad = 1e-2, 5e-2
    else:
        # Default fallback
        rtol_grad, atol_grad = 1e-5, 1e-5

    is_q_grad_close = torch.allclose(
        grad_q_einsum, grad_q_pytorch, rtol=rtol_grad, atol=atol_grad
    )
    is_k_grad_close = torch.allclose(
        grad_k_einsum, grad_k_pytorch, rtol=rtol_grad, atol=atol_grad
    )
    is_v_grad_close = torch.allclose(
        grad_v_einsum, grad_v_pytorch, rtol=rtol_grad, atol=atol_grad
    )

    logger.info("Gradient equivalence:")
    logger.info(f"Query grad close: {is_q_grad_close}")
    logger.info(f"Key grad close: {is_k_grad_close}")
    logger.info(f"Value grad close: {is_v_grad_close}")

    assert (
        is_close
    ), f"Forward pass outputs not close. Max diff: {max_diff:.6e}, Mean diff: {mean_diff:.6e}"
    assert is_q_grad_close, "Query gradients not close"
    assert is_k_grad_close, "Key gradients not close"
    assert is_v_grad_close, "Value gradients not close"

In [52]:
test_attn_cuda = TestAttentionBackendsCUDA()
test_attn_cpu = TestAttentionBackendsCPU()

In [53]:
test_attn_cuda.test_cuda_no_amp()
test_attn_cuda.test_cuda_amp_float16()
test_attn_cuda.test_cuda_amp_bfloat16()

INFO:test:================================================================================
INFO:test:CONFIG: batch=32, heads=8, seq=720, dim=256
INFO:test:Device: cuda, AMP: False (N/A)
INFO:test:================================================================================
INFO:test:EQUIVALENCE CHECK
INFO:test:Dtype: torch.float32
INFO:test:Tolerance: rtol=1e-05, atol=1e-05
INFO:test:All close: True
INFO:test:Max diff: 1.609325e-06
INFO:test:Mean diff: 4.146193e-08
INFO:test:================================================================================
INFO:test:FORWARD PASS
INFO:test:Einsum backend: 36.24 ms | 1827.50 MB
INFO:test:PyTorch backend: 11.32 ms | 353.75 MB
INFO:test:Speedup: 3.20x
INFO:test:Memory saved: 1473.75 MB (80.6%)
INFO:test:================================================================================
INFO:test:BACKWARD PASS
INFO:test:Einsum: 135.90 ms | 2423.75 MB
INFO:test:PyTorch: 40.89 ms | 538.22 MB
INFO:test:Speedup: 3.32x
INFO:test:Memory saved: 1885

In [56]:
test_attn_cpu.test_cpu_no_amp()
test_attn_cpu.test_cpu_amp_float16()
test_attn_cpu.test_cpu_amp_bfloat16()

INFO:test:================================================================================
INFO:test:CONFIG: batch=32, heads=8, seq=720, dim=256
INFO:test:Device: cpu, AMP: False (N/A)
INFO:test:================================================================================
INFO:test:EQUIVALENCE CHECK
INFO:test:Dtype: torch.float32
INFO:test:Tolerance: rtol=1e-05, atol=1e-05
INFO:test:All close: True
INFO:test:Max diff: 8.940697e-07
INFO:test:Mean diff: 1.890439e-08
INFO:test:================================================================================
INFO:test:FORWARD PASS
INFO:test:Einsum backend: 208.53 ms | 0.00 MB
INFO:test:PyTorch backend: 63.43 ms | 0.00 MB
INFO:test:Speedup: 3.29x
INFO:test:================================================================================
INFO:test:BACKWARD PASS
INFO:test:Einsum: 608.67 ms | 0.00 MB
INFO:test:PyTorch: 185.61 ms | 0.00 MB
INFO:test:Speedup: 3.28x
INFO:test:======================================================================